In [0]:
-- Fetch the qdp_organisation_customers document data product created by 01. Quantexa Processing of Raw Data in Bronze

-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;


-- 01-2. Create default start date for all data inserts

DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;


-- 01-3. Create default end date for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS default_end_date;

DECLARE default_end_date STRING;

SET  VAR default_end_date = '31-12-2999';

SELECT default_end_date;


 
-- 2. Remove all Address records for the Tennant
DELETE FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customers s
WHERE s.organisation_customer_id IN (
  SELECT DISTINCT organisation_customer_id
  FROM demo_banking_silver.qdp_organisation_customers.hub_organisation_customers
  WHERE tennant_id = :p_tennant_id)
;

DELETE FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customer_risk_scores s
WHERE s.organisation_customer_id IN (
  SELECT DISTINCT organisation_customer_id
  FROM demo_banking_silver.qdp_organisation_customers.hub_organisation_customers
  WHERE tennant_id = :p_tennant_id)
;
/**
DELETE FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customer_warning_signals s
WHERE s.organisation_customer_id IN (
  SELECT DISTINCT organisation_customer_id
  FROM demo_banking_silver.qdp_organisation_customers.hub_organisation_customers
  WHERE tennant_id = :p_tennant_id)
;
**/
DELETE FROM demo_banking_silver.qdp_organisation_customers.hub_organisation_customers WHERE tennant_id = :p_tennant_id;


-- 3. Assign the correct address_id or leave as defaulted zero value if does not exist
CREATE OR REPLACE TEMP VIEW organisation_customers_correct_types_addresses AS
SELECT 
    q.*,
    COALESCE(haddr.address_id, 0) AS address_id
FROM demo_banking_silver.qdp_from_quantexa_banking_supply_chain.qdp_organisation_customers q
LEFT JOIN demo_banking_silver.qdp_locations_all.hub_address haddr
ON q.address_entity_id = haddr.address_entity_id
;

SELECT * from organisation_customers_correct_types_addresses;



-- 4.  Populate the hub_organisation_customer records
INSERT INTO demo_banking_silver.qdp_organisation_customers.hub_organisation_customers
(organisation_customer_entity_id, tennant_id, organisation_entity_id, address_entity_id, period_start, period_end)
SELECT organisation_customer_entity_id, tennant_id, organisation_entity_id, address_entity_id, period_start, period_end
FROM organisation_customers_correct_types_addresses;

SELECT * FROM demo_banking_silver.qdp_organisation_customers.hub_organisation_customers;


-- 5.  Populate the sat_organisation_customer records
INSERT INTO demo_banking_silver.qdp_organisation_customers.sat_organisation_customers (
--     sat_organisation_customer_id,
     organisation_customer_id,
     organisation_id,
     address_id,
     customer_name,
     load_datetime, 
     record_source_id, 
     data_source_code,
     data_source_concept_id,
     data_source_raw_code,
     data_source_raw_concept_id,
     type_code,
     type_concept_id,
     type_raw_code,
     type_raw_concept_id,
     line_of_business_code,
     line_of_business_concept_id,
     line_of_business_raw_code,
     line_of_business_raw_concept_id,
     importance_code,
     importance_concept_id,
     importance_raw_code,
     importance_raw_concept_id,
     status_code,
     status_concept_id,
     status_raw_code,
     status_raw_concept_id,
     risk_rating,
     risk_rating_raw,
     risk_rating_code,
     risk_rating_concept_id,
     risk_rating_raw_code,
     risk_rating_raw_concept_id,
     is_customer_alarm,
     is_offboarded,
     period_start,
     period_end
    )
SELECT
--  monotonically_increasing_id() AS sat_organisation_customer_id,
  h.organisation_customer_id AS organisation_customer_id,
  org.organisation_id AS organisation_id,
  q.address_id AS address_id,
  customer_name AS customer_name,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.data_source_code,
  q.data_source_concept_id,
  q.data_source_raw_code,
  q.data_source_raw_concept_id,
  q.type_code,
  q.type_concept_id,
  q.type_raw_code,
  q.type_raw_concept_id,
  q.line_of_business_code,
  q.line_of_business_concept_id,
  q.line_of_business_raw_code,
  q.line_of_business_raw_concept_id,
  q.importance_code,
  q.importance_concept_id,
  q.importance_raw_code,
  q.importance_raw_concept_id,
  q.status_code,
  q.status_concept_id,
  q.status_raw_code,
  q.status_raw_concept_id,
  q.risk_rating,
  q.risk_rating_raw,
  q.risk_rating_code,
  q.risk_rating_concept_id,
  q.risk_rating_raw_code,
  q.risk_rating_raw_concept_id,
  q.is_customer_alarm,
  q.is_offboarded,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM organisation_customers_correct_types_addresses q
JOIN demo_banking_silver.qdp_organisation_customers.hub_organisation_customers h
  ON h.organisation_customer_entity_id = q.organisation_customer_entity_id
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation org
  ON org.organisation_entity_id = q.organisation_entity_id
;
  
SELECT * FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customers;


